# Selenium ?? ?? ??

- ??: YouTube ????? ?? ??? ????? ?? ?????.
- ??: ?? ???, Selenium WebDriver
- ??: ?? ???URL???? ? ?? ???
- ?? ??: ??? ??? ?? ?? ???? ?? ????? ?????.


In [ ]:
# !pip install pytube

In [ ]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
from openpyxl import Workbook
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from pytube import YouTube
import datetime
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm
import time
import re
import requests
import random

# 필요 클래스 정의

In [ ]:
# 좀 더 빨리 스크롤 내리는 방법 인용 # https://manyda.tistory.com/10
#Queue의 기본적인 기능 구현
class Queue():
    def __init__(self, maxsize):
        self.queue = []
        self.maxsize = maxsize
        
    # Queue에 Data 넣음
    def enqueue(self, data):
        self.queue.append(data)

    # Queue에 가장 먼저 들어온 Data 내보냄
    def dequeue(self):
        dequeue_object = None
        if self.isEmpty():
            print("Queue is Empty")
        else:
            dequeue_object = self.queue[0]
            self.queue = self.queue[1:]
        return dequeue_object
    
    # Queue에 가장 먼저들어온 Data return
    def peek(self):
        peek_object = None
        if self.isEmpty():
            print("Queue is Empty")
        else:
            peek_object = self.queue[0]
        return peek_object
    
    # Queue가 비어있는지 확인
    def isEmpty(self):
        is_empty = False
        if len(self.queue) == 0:
            is_empty = True
        return is_empty
    
    # Queue의 Size가 Max Size를 초과하는지 확인
    def isMaxSizeOver(self):
        queue_size = len(self.queue)
        if (queue_size > self.maxsize):
            return False
        else :
            return True

# 유튜버 영상들 정보 수집

## 특정 유튜버 채널 들어가기

In [ ]:
df = pd.DataFrame(columns=['제목', 'URL', '조회수', '업데이트 날짜', '영상 길이', '영상 좋아요 수']) # 저장 데이터프레임 생성

# 크롬 웹브라우저 실행
# driver = webdriver.Chrome(ChromeDriverManager().install())
driver = webdriver.Chrome("chromedriver-win64/chromedriver.exe")
driver.maximize_window()

keyword = '회사원A' # 검색하고자 하는 유튜버 입력

# 키워드로 접속
driver.get(f"https://www.youtube.com/results?search_query={keyword}")
# time.sleep(2) # 오류나면 잠시 기다리는 코드 추가해보기
driver.find_element(By.XPATH, '//*[@id="avatar-section"]/a').click() # 채널 클릭
time.sleep(1)
driver.find_element(By.XPATH, '//*[@id="tabsContent"]/tp-yt-paper-tab[2]').click() # 동영상 목록 클릭

## 스크롤 내리기 

In [ ]:
#down the scroll
body = driver.find_element(By.TAG_NAME, 'body')
last_page_height = driver.execute_script("return document.documentElement.scrollHeight")

# max size 50의 Queue 생성
# 0.1sec * 50 = 5sec 동안 Scroll 업데이트가 없으면 스크롤 내리기 종료
szQ = Queue(50)
enqueue_count = 0

i = 0
while True:
    # Scroll 내리기
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
    
    # Scroll Height를 가져오는 주기
    time.sleep(0.1)
    new_page_height = driver.execute_script("return document.documentElement.scrollHeight")
    
    # Queue가 꽉 차는 경우 스크롤 내리기 종료
    if(enqueue_count > szQ.maxsize):
        break
    
    # 첫 Loop 수행 (Queue가 비어있는 경우) 예외 처리
    if(szQ.isEmpty()) :
        szQ.enqueue(new_page_height)
        enqueue_count += 1
        
    # Queue에 가장 먼저 들어온 데이터와 새로 업데이트 된 Scroll Height를 비교함
    # 같으면 그대로 Enqueue, 다르면 Queue의 모든 Data를 Dequeue 후 새로운 Scroll Height를 Enqueue 함.    
    else :
        if(szQ.peek() == new_page_height) :
            szQ.enqueue(new_page_height)
            enqueue_count += 1
        else :
            szQ.enqueue(new_page_height)
            for z in range(enqueue_count) :
                szQ.dequeue()
            enqueue_count = 1
    
    # 이 부분은 모든 영상을 수집하기에는 무리가 있어서 추가한 부분 20을 더 작은 수나 더 큰 수로 바꿔서 실행해도 문제없음
    if i > 20:
        break
    else:
        i += 1

## 채널의 영상 정보들 수집

In [ ]:
titles = driver.find_elements(By.XPATH, '//*[@id="video-title-link"]') 
for title in titles:
    main_title = title.get_property("title")
    url = title.get_property("href")
    df.loc[len(df), ['제목', 'URL']] =  [main_title, url]

In [ ]:
df

## 라이브러리를 이용한 영상정보 수집

In [ ]:
import tqdm


for i in tqdm.tqdm(range(len(df))):

    url = df.iloc[i,1]
    tube = YouTube(url)

    view = tube.views # 조회수 불러오기
    update_dates = str(tube.publish_date)  # 업데이트 날짜 불러와서 str 형태로 변환
    update_date = update_dates.split(" ")  # 업데이트 날짜 string을 공백으로 분리

    # 업데이트 날짜는 0000-00-00 00-00-00 형태인데 앞의 날짜부분만 추출하려 합니다.
    length_second = int(tube.length) # 유튜브 영상의 길이를 불러옴(문자 형태인것 같아서 int로 변환) / 문자가 아닐 수도...
    length = str(datetime.timedelta(seconds=length_second)) # datetime 모듈의 timedelta 함수로 초를 시:분:초 로 변환
    
    df.iloc[i, 2] = view 
    df.iloc[i, 3] = update_date[0] # 추출한 날짜 중 앞의 날짜부분만 추출
    df.iloc[i, 4] = length
df

In [ ]:
df = df[df['업데이트 날짜'] > '2022-08-01']
df

In [ ]:
# https://pytube.io/
print("제목 : ", tube.title)
print("길이 : ", tube.length)
print("게시자 : ", tube.author)
print("게시날짜 : ", tube.publish_date)
print("조회수 : ", tube.views)
print("키워드 : ", tube.keywords)
print("설명 : ", tube.description)
print("썸네일 : ", tube.thumbnail_url)

In [ ]:
driver = webdriver.Chrome("chromedriver-win64/chromedriver.exe")

# 특정 영상의 댓글 수집

In [ ]:
# #set option of selenium
# options = webdriver.ChromeOptions()
# options.add_argument('window-size=1920x1080')
# options.add_argument('disable-gpu')
# options.add_argument('user')
# options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_12_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/61.0.3163.100 Safari/537.36")
# options.add_argument("lang=ko_KR")
# # driver = webdriver.Chrome(ChromeDriverManager().install(), chrome_options=options)
# driver = webdriver.Chrome("chromedriver-win64/chromedriver.exe")

import os
# 수집한 영상들에 대해서 반복해서 댓글 수집
for video_num in range(len(df)):
    url = df.iloc[video_num+2,1]
    driver.get(url)

    #페이지 Open 후 기다리는 시간
    time.sleep(7)
    
    # 영상 좋아요 수
    temp = driver.find_element(By.XPATH, '//*[@id="segmented-like-button"]/ytd-toggle-button-renderer/yt-button-shape/button')
    good = temp.get_attribute('aria-label')
    good_num = re.findall(r'\d{1,3}(?:,\d{3})*', good)
    if good_num:
        good_num = good_num[0].replace(',', '')  # 쉼표 제거
        df.iloc[video_num, 5] = good_num



    #down the scroll
    body = driver.find_element(By.TAG_NAME, 'body')
    last_page_height = driver.execute_script("return document.documentElement.scrollHeight")

    # max size 50의 Queue 생성
    # 0.1sec * 50 = 5sec 동안 Scroll 업데이트가 없으면 스크롤 내리기 종료
    szQ = Queue(3)
    enqueue_count = 0
    time.sleep(5)
    i = 0
    while True:
        # Scroll 내리기
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 10);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 2);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);") 
        # Scroll Height를 가져오는 주기
        time.sleep(0.1)
        new_page_height = driver.execute_script("return document.documentElement.scrollHeight")
        
        # Queue가 꽉 차는 경우 스크롤 내리기 종료
        if(enqueue_count > szQ.maxsize):
            break
        
        # 첫 Loop 수행 (Queue가 비어있는 경우) 예외 처리
        if(szQ.isEmpty()) :
            szQ.enqueue(new_page_height)
            enqueue_count += 1
            
        # Queue에 가장 먼저 들어온 데이터와 새로 업데이트 된 Scroll Height를 비교함
        # 같으면 그대로 Enqueue, 다르면 Queue의 모든 Data를 Dequeue 후 새로운 Scroll Height를 Enqueue 함.    
        else :
            if(szQ.peek() == new_page_height) :
                szQ.enqueue(new_page_height)
                enqueue_count += 1
            else :
                szQ.enqueue(new_page_height)
                for z in range(enqueue_count) :
                    szQ.dequeue()
                enqueue_count = 1
        time.sleep(5)
        if i > 30:
            break
        else:
            i += 1


    time.sleep(1.5)
           
    print('스크롤 끝')
    time.sleep(5)   
    # 댓글 정보 불러오기
    comments_list = driver.find_elements(By.XPATH, '//*[@id="comment"]')

    comment_data = pd.DataFrame({'youtube_id':[],
                                'comment':[],
                                'like_num':[], 'comment_date':[]})
    html_s0 = driver.page_source
    html_s = BeautifulSoup(html_s0,'html.parser')

    comment0 = html_s.find_all('ytd-comment-renderer',{'class':'style-scope ytd-comment-thread-renderer'})
    print('댓글정보 불러오기 시작')
    for i in range(len(comment0)):
        

        # 댓글
        comment = comment0[i].find('yt-formatted-string',{'id':'content-text','class':'style-scope ytd-comment-renderer'}).text
        # 댓글의 좋아요 수
        try:
            aa = comment0[i].find('span',{'id':'vote-count-left'}).text
            #정규표현식으로 숫자만 추출하는 것은 정규표현식에 대한 공부를 더 한 뒤 해결
            #re.findall('[0-9]',aa)
            #"".join(re.findall('[0-9]',aa)) -> 리스트 내부의 문자열의 합
            like_num = "".join(re.findall('[0-9]',aa))
            like_num = int(like_num)
        except:
            like_num = 0

        # 댓글 작성자 id    
        bb = comment0[i].find('a',{'id':'author-text'}).find('span').text
        youtube_id = "".join(re.findall('[가-힣0-9a-zA-Z]',bb))

        # 댓글 작성 날짜
        comment_date = comment0[i].find('yt-formatted-string',{'class':'published-time-text style-scope ytd-comment-renderer'}).text
        
        # 데이터프레임에 데이터 저장
        comment_data.loc[len(comment_data), :] = [youtube_id, comment, like_num, comment_date]

    comment_data.index = range(len(comment_data))
    # 파일 저장
    if not os.path.exists(keyword):
        os.makedirs(keyword)

    comment_data.to_csv(f'./data/{keyword}/{keyword}_{video_num}_comment_data.csv')
    print(f'{keyword}_{video_num}_comment_data.csv 저장완료')
    

In [ ]:
comment_data

# 전체 반복문

In [ ]:
temp_list = pd.read_excel('./data/뷰티_리스트.xlsx')
temp_list.iloc[197:200, 0].to_list()    

In [ ]:
temp_list.iloc[197:200, 0].to_list() + ['레삐디오(RepitProfessional)', '몽키TV', '패션-정찬양']

In [ ]:
temp_list.iloc[180:200, 0].to_list() 

In [ ]:
import os
youtuber_list = temp_list['0'].to_list()[3:]

# 크롬 웹브라우저 실행
driver = webdriver.Chrome(ChromeDriverManager().install())
#driver = webdriver.Chrome("chromedriver-win64/chromedriver.exe")
driver.maximize_window()

for youtuber in youtuber_list:
    df = pd.DataFrame(columns=['제목', 'URL', '조회수', '업데이트 날짜', '영상 길이', '영상 좋아요 수']) # 저장 데이터프레임 생성



    # 키워드로 접속
    driver.get(f"https://www.youtube.com/results?search_query={youtuber}")
    time.sleep(1) # 오류나면 잠시 기다리는 코드 추가해보기
    driver.find_element(By.XPATH, '//*[@id="avatar-section"]/a').click() # 채널 클릭
    time.sleep(1)
    driver.find_element(By.XPATH, '//*[@id="tabsContent"]/tp-yt-paper-tab[2]').click() # 동영상 목록 클릭
    


    #down the scroll
    body = driver.find_element(By.TAG_NAME, 'body')
    last_page_height = driver.execute_script("return document.documentElement.scrollHeight")

    # max size 50의 Queue 생성
    # 0.1sec * 50 = 5sec 동안 Scroll 업데이트가 없으면 스크롤 내리기 종료
    szQ = Queue(50)
    enqueue_count = 0

    i = 0
    while True:
        # Scroll 내리기
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        
        # Scroll Height를 가져오는 주기
        time.sleep(0.1)
        new_page_height = driver.execute_script("return document.documentElement.scrollHeight")
        
        # Queue가 꽉 차는 경우 스크롤 내리기 종료
        if(enqueue_count > szQ.maxsize):
            break
        
        # 첫 Loop 수행 (Queue가 비어있는 경우) 예외 처리
        if(szQ.isEmpty()) :
            szQ.enqueue(new_page_height)
            enqueue_count += 1
            
        # Queue에 가장 먼저 들어온 데이터와 새로 업데이트 된 Scroll Height를 비교함
        # 같으면 그대로 Enqueue, 다르면 Queue의 모든 Data를 Dequeue 후 새로운 Scroll Height를 Enqueue 함.    
        else :
            if(szQ.peek() == new_page_height) :
                szQ.enqueue(new_page_height)
                enqueue_count += 1
            else :
                szQ.enqueue(new_page_height)
                for z in range(enqueue_count) :
                    szQ.dequeue()
                enqueue_count = 1
        
        # 이 부분은 모든 영상을 수집하기에는 무리가 있어서 추가한 부분 20을 더 작은 수나 더 큰 수로 바꿔서 실행해도 문제없음
        print(i)
        if i > 6:
            break
        else:
            i += 1

    titles = driver.find_elements(By.XPATH, '//*[@id="video-title-link"]') 
    for title in titles:
        main_title = title.get_property("title")
        url = title.get_property("href")
        df.loc[len(df), ['제목', 'URL']] =  [main_title, url]

    
    for i in range(len(df)):

        url = df.iloc[i,1]
        tube = YouTube(url)

        view = tube.views # 조회수 불러오기
        update_dates = str(tube.publish_date)  # 업데이트 날짜 불러와서 str 형태로 변환
        update_date = update_dates.split(" ")  # 업데이트 날짜 string을 공백으로 분리

        # 업데이트 날짜는 0000-00-00 00-00-00 형태인데 앞의 날짜부분만 추출하려 합니다.
        length_second = int(tube.length) # 유튜브 영상의 길이를 불러옴(문자 형태인것 같아서 int로 변환) / 문자가 아닐 수도...
        length = str(datetime.timedelta(seconds=length_second)) # datetime 모듈의 timedelta 함수로 초를 시:분:초 로 변환
        
        df.iloc[i, 2] = view 
        df.iloc[i, 3] = update_date[0] # 추출한 날짜 중 앞의 날짜부분만 추출
        df.iloc[i, 4] = length
        if update_date[0] < '2022-08-01':
            break

    df = df[df['업데이트 날짜'] > '2022-08-01']
    df = df[df['업데이트 날짜'] < '2023-09-01']
    try:
        os.makedirs(f'./new_data/{youtuber}')
    except:
        pass
    df.to_csv(f'./new_data/{youtuber}/{youtuber}_video_data.csv')
    print(f'{youtuber}_video_data.csv 저장완료')
    


    import os
    # 수집한 영상들에 대해서 반복해서 댓글 수집
    for video_num in range(len(df)):
        url = df.iloc[video_num,1]
        driver.get(url)

        #페이지 Open 후 기다리는 시간
        time.sleep(3)
        
        # 영상 좋아요 수
        temp = driver.find_element(By.XPATH, '//*[@id="segmented-like-button"]/ytd-toggle-button-renderer/yt-button-shape/button')
        good = temp.get_attribute('aria-label')
        good_num = re.findall(r'\d{1,3}(?:,\d{3})*', good)
        if good_num:
            good_num = good_num[0].replace(',', '')  # 쉼표 제거
            df.iloc[video_num, 5] = good_num
            print(good_num)



        #down the scroll
        body = driver.find_element(By.TAG_NAME, 'body')
        last_page_height = driver.execute_script("return document.documentElement.scrollHeight")
        # max size 50의 Queue 생성
        # 0.1sec * 50 = 5sec 동안 Scroll 업데이트가 없으면 스크롤 내리기 종료
        szQ = Queue(3)
        enqueue_count = 0
        time.sleep(3)
        i = 0
        while True:
            # Scroll 내리기
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 10);")
            time.sleep(1)
            # driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 2);")
            # time.sleep(1)
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);") 
            # Scroll Height를 가져오는 주기
            time.sleep(1)
            new_page_height = driver.execute_script("return document.documentElement.scrollHeight")
            
            # Queue가 꽉 차는 경우 스크롤 내리기 종료
            if(enqueue_count > szQ.maxsize):
                break
            
            # 첫 Loop 수행 (Queue가 비어있는 경우) 예외 처리
            if(szQ.isEmpty()) :
                szQ.enqueue(new_page_height)
                enqueue_count += 1
                
            # Queue에 가장 먼저 들어온 데이터와 새로 업데이트 된 Scroll Height를 비교함
            # 같으면 그대로 Enqueue, 다르면 Queue의 모든 Data를 Dequeue 후 새로운 Scroll Height를 Enqueue 함.    
            else :
                if(szQ.peek() == new_page_height) :
                    szQ.enqueue(new_page_height)
                    enqueue_count += 1
                else :
                    szQ.enqueue(new_page_height)
                    for z in range(enqueue_count) :
                        szQ.dequeue()
                    enqueue_count = 1
            time.sleep(0.5)
            if i > 15:
                break
            else:
                print(i)
                i += 1
                   
        print('스크롤 끝')
        time.sleep(2)   
        # 댓글 정보 불러오기
        comments_list = driver.find_elements(By.XPATH, '//*[@id="comment"]')

        comment_data = pd.DataFrame({'youtube_id':[],
                                    'comment':[],
                                    'like_num':[], 'comment_date':[]})
        html_s0 = driver.page_source
        html_s = BeautifulSoup(html_s0,'html.parser')

        comment0 = html_s.find_all('ytd-comment-renderer',{'class':'style-scope ytd-comment-thread-renderer'})
        print('댓글정보 불러오기 시작')
        for i in range(len(comment0)):
            

            # 댓글
            comment = comment0[i].find('yt-formatted-string',{'id':'content-text','class':'style-scope ytd-comment-renderer'}).text
            # 댓글의 좋아요 수
            try:
                aa = comment0[i].find('span',{'id':'vote-count-left'}).text
                #정규표현식으로 숫자만 추출하는 것은 정규표현식에 대한 공부를 더 한 뒤 해결
                #re.findall('[0-9]',aa)
                #"".join(re.findall('[0-9]',aa)) -> 리스트 내부의 문자열의 합
                like_num = "".join(re.findall('[0-9]',aa))
                like_num = int(like_num)
            except:
                like_num = 0

            # 댓글 작성자 id    
            bb = comment0[i].find('a',{'id':'author-text'}).find('span').text
            youtube_id = "".join(re.findall('[가-힣0-9a-zA-Z]',bb))

            # 댓글 작성 날짜
            comment_date = comment0[i].find('yt-formatted-string',{'class':'published-time-text style-scope ytd-comment-renderer'}).text
            
            # 데이터프레임에 데이터 저장
            comment_data.loc[len(comment_data), :] = [youtube_id, comment, like_num, comment_date]

        comment_data.index = range(len(comment_data))
        # 파일 저장


        comment_data.to_csv(f'./new_data/{youtuber}/{youtuber}_{video_num}_comment_data.csv')
        print(f'{youtuber}_{video_num}_comment_data.csv 저장완료')
    df.to_csv(f'./new_data/{youtuber}/{youtuber}_video_data.csv')
    print(f'{youtuber}_video_data.csv 저장완료')
        

In [ ]:
youtuber_list = ['회사원A', '씬님', '기우쌤'] 

for youtuber in youtuber_list:
    df = pd.DataFrame(columns=['제목', 'URL', '조회수', '업데이트 날짜', '영상 길이', '영상 좋아요 수']) # 저장 데이터프레임 생성

    # 크롬 웹브라우저 실행
    # driver = webdriver.Chrome(ChromeDriverManager().install())
    driver = webdriver.Chrome("chromedriver-win64/chromedriver.exe")
    driver.maximize_window()

    # 키워드로 접속
    driver.get(f"https://www.youtube.com/results?search_query={youtuber}")
    # time.sleep(2) # 오류나면 잠시 기다리는 코드 추가해보기
    driver.find_element(By.XPATH, '//*[@id="avatar-section"]/a').click() # 채널 클릭
    time.sleep(3)
    driver.find_element(By.XPATH, '//*[@id="tabsContent"]/tp-yt-paper-tab[2]').click() # 동영상 목록 클릭
    


    #down the scroll
    body = driver.find_element(By.TAG_NAME, 'body')
    last_page_height = driver.execute_script("return document.documentElement.scrollHeight")

    # max size 50의 Queue 생성
    # 0.1sec * 50 = 5sec 동안 Scroll 업데이트가 없으면 스크롤 내리기 종료
    szQ = Queue(50)
    enqueue_count = 0
    time.sleep(3)
    i = 0
    while True:
        # Scroll 내리기
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        
        # Scroll Height를 가져오는 주기
        time.sleep(0.1)
        new_page_height = driver.execute_script("return document.documentElement.scrollHeight")
        
        # Queue가 꽉 차는 경우 스크롤 내리기 종료
        if(enqueue_count > szQ.maxsize):
            break
        
        # 첫 Loop 수행 (Queue가 비어있는 경우) 예외 처리
        if(szQ.isEmpty()) :
            szQ.enqueue(new_page_height)
            enqueue_count += 1
            
        # Queue에 가장 먼저 들어온 데이터와 새로 업데이트 된 Scroll Height를 비교함
        # 같으면 그대로 Enqueue, 다르면 Queue의 모든 Data를 Dequeue 후 새로운 Scroll Height를 Enqueue 함.    
        else :
            if(szQ.peek() == new_page_height) :
                szQ.enqueue(new_page_height)
                enqueue_count += 1
            else :
                szQ.enqueue(new_page_height)
                for z in range(enqueue_count) :
                    szQ.dequeue()
                enqueue_count = 1
        
        # 이 부분은 모든 영상을 수집하기에는 무리가 있어서 추가한 부분 20을 더 작은 수나 더 큰 수로 바꿔서 실행해도 문제없음
        print(i)
        if i > 6:
            break
        else:
            i += 1

    titles = driver.find_elements(By.XPATH, '//*[@id="video-title-link"]') 
    for title in titles:
        main_title = title.get_property("title")
        url = title.get_property("href")
        df.loc[len(df), ['제목', 'URL']] =  [main_title, url]

    
    for i in range(len(df)):

        url = df.iloc[i,1]
        tube = YouTube(url)

        view = tube.views # 조회수 불러오기
        update_dates = str(tube.publish_date)  # 업데이트 날짜 불러와서 str 형태로 변환
        update_date = update_dates.split(" ")  # 업데이트 날짜 string을 공백으로 분리

        # 업데이트 날짜는 0000-00-00 00-00-00 형태인데 앞의 날짜부분만 추출하려 합니다.
        length_second = int(tube.length) # 유튜브 영상의 길이를 불러옴(문자 형태인것 같아서 int로 변환) / 문자가 아닐 수도...
        length = str(datetime.timedelta(seconds=length_second)) # datetime 모듈의 timedelta 함수로 초를 시:분:초 로 변환
        
        df.iloc[i, 2] = view 
        df.iloc[i, 3] = update_date[0] # 추출한 날짜 중 앞의 날짜부분만 추출
        df.iloc[i, 4] = length
        if update_date[0] < '2022-08-01':
            break

    df = df[df['업데이트 날짜'] > '2022-08-01']
    df = df[df['업데이트 날짜'] < '2023-09-01']
    try:
        os.makedirs(f'./data/{youtuber}')
    except:
        pass
    df.to_csv(f'./data/{youtuber}/{youtuber}_video_data.csv')
    print(f'{youtuber}_video_data.csv 저장완료')
    


    import os
    # 수집한 영상들에 대해서 반복해서 댓글 수집
    for video_num in range(len(df)):
        url = df.iloc[video_num,1]
        driver.get(url)

        #페이지 Open 후 기다리는 시간
        time.sleep(5)
        
        # 영상 좋아요 수
        temp = driver.find_element(By.XPATH, '//*[@id="segmented-like-button"]/ytd-toggle-button-renderer/yt-button-shape/button')
        good = temp.get_attribute('aria-label')
        good_num = re.findall(r'\d{1,3}(?:,\d{3})*', good)
        if good_num:
            good_num = good_num[0].replace(',', '')  # 쉼표 제거
            df.iloc[video_num, 5] = good_num


        # comment_data.to_csv(f'./data/{youtuber}/{youtuber}_{video_num}_comment_data.csv')
        # print(f'{youtuber}_{video_num}_comment_data.csv 저장완료')
    df.to_csv(f'./data/{youtuber}/{youtuber}_video_data.csv')
    print(f'{youtuber}_video_data.csv 저장완료')
        

In [ ]:
a = pd.read_csv('./data/씬님/씬님_video_data.csv')
a

In [ ]:
# #set option of selenium
# options = webdriver.ChromeOptions()
# options.add_argument('window-size=1920x1080')
# options.add_argument('disable-gpu')
# options.add_argument('user')
# options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_12_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/61.0.3163.100 Safari/537.36")
# options.add_argument("lang=ko_KR")
# # driver = webdriver.Chrome(ChromeDriverManager().install(), chrome_options=options)
# driver = webdriver.Chrome("chromedriver-win64/chromedriver.exe")

import os
# 수집한 영상들에 대해서 반복해서 댓글 수집
for video_num in range(len(df)):
    url = df.iloc[video_num+2,1]
    driver.get(url)

    #페이지 Open 후 기다리는 시간
    time.sleep(7)
    
    # 영상 좋아요 수
    temp = driver.find_element(By.XPATH, '//*[@id="segmented-like-button"]/ytd-toggle-button-renderer/yt-button-shape/button')
    good = temp.get_attribute('aria-label')
    good_num = re.findall(r'\d{1,3}(?:,\d{3})*', good)
    if good_num:
        good_num = good_num[0].replace(',', '')  # 쉼표 제거
        df.iloc[video_num, 5] = good_num



    #down the scroll
    body = driver.find_element(By.TAG_NAME, 'body')
    last_page_height = driver.execute_script("return document.documentElement.scrollHeight")

    # max size 50의 Queue 생성
    # 0.1sec * 50 = 5sec 동안 Scroll 업데이트가 없으면 스크롤 내리기 종료
    szQ = Queue(3)
    enqueue_count = 0
    time.sleep(5)
    i = 0
    while True:
        # Scroll 내리기
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 10);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 2);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);") 
        # Scroll Height를 가져오는 주기
        time.sleep(0.1)
        new_page_height = driver.execute_script("return document.documentElement.scrollHeight")
        
        # Queue가 꽉 차는 경우 스크롤 내리기 종료
        if(enqueue_count > szQ.maxsize):
            break
        
        # 첫 Loop 수행 (Queue가 비어있는 경우) 예외 처리
        if(szQ.isEmpty()) :
            szQ.enqueue(new_page_height)
            enqueue_count += 1
            
        # Queue에 가장 먼저 들어온 데이터와 새로 업데이트 된 Scroll Height를 비교함
        # 같으면 그대로 Enqueue, 다르면 Queue의 모든 Data를 Dequeue 후 새로운 Scroll Height를 Enqueue 함.    
        else :
            if(szQ.peek() == new_page_height) :
                szQ.enqueue(new_page_height)
                enqueue_count += 1
            else :
                szQ.enqueue(new_page_height)
                for z in range(enqueue_count) :
                    szQ.dequeue()
                enqueue_count = 1
        time.sleep(5)
        if i > 30:
            break
        else:
            i += 1


    time.sleep(1.5)
           
    print('스크롤 끝')
    time.sleep(5)   
    # 댓글 정보 불러오기
    comments_list = driver.find_elements(By.XPATH, '//*[@id="comment"]')

    comment_data = pd.DataFrame({'youtube_id':[],
                                'comment':[],
                                'like_num':[], 'comment_date':[]})
    html_s0 = driver.page_source
    html_s = BeautifulSoup(html_s0,'html.parser')

    comment0 = html_s.find_all('ytd-comment-renderer',{'class':'style-scope ytd-comment-thread-renderer'})
    print('댓글정보 불러오기 시작')
    for i in range(len(comment0)):
        

        # 댓글
        comment = comment0[i].find('yt-formatted-string',{'id':'content-text','class':'style-scope ytd-comment-renderer'}).text
        # 댓글의 좋아요 수
        try:
            aa = comment0[i].find('span',{'id':'vote-count-left'}).text
            #정규표현식으로 숫자만 추출하는 것은 정규표현식에 대한 공부를 더 한 뒤 해결
            #re.findall('[0-9]',aa)
            #"".join(re.findall('[0-9]',aa)) -> 리스트 내부의 문자열의 합
            like_num = "".join(re.findall('[0-9]',aa))
            like_num = int(like_num)
        except:
            like_num = 0

        # 댓글 작성자 id    
        bb = comment0[i].find('a',{'id':'author-text'}).find('span').text
        youtube_id = "".join(re.findall('[가-힣0-9a-zA-Z]',bb))

        # 댓글 작성 날짜
        comment_date = comment0[i].find('yt-formatted-string',{'class':'published-time-text style-scope ytd-comment-renderer'}).text
        
        # 데이터프레임에 데이터 저장
        comment_data.loc[len(comment_data), :] = [youtube_id, comment, like_num, comment_date]

    comment_data.index = range(len(comment_data))
    # 파일 저장
    if not os.path.exists(keyword):
        os.makedirs(keyword)

    comment_data.to_csv(f'./data/{keyword}/{keyword}_{video_num}_comment_data.csv')
    print(f'{keyword}_{video_num}_comment_data.csv 저장완료')
    

In [ ]:
df = pd.read_csv('./data/회사원A/회사원A_video_data.csv', index_col=0)

In [ ]:
for video_num in range(len(df)):
    url = df.iloc[video_num+1,1]
    driver.get(url)

    #페이지 Open 후 기다리는 시간
    time.sleep(10)
    
    # 영상 좋아요 수
    temp = driver.find_element(By.XPATH, '//*[@id="segmented-like-button"]/ytd-toggle-button-renderer/yt-button-shape/button')
    good = temp.get_attribute('aria-label')
    good_num = re.findall(r'\d{1,3}(?:,\d{3})*', good)
    if good_num:
        good_num = good_num[0].replace(',', '')  # 쉼표 제거
        df.iloc[video_num, 5] = good_num


    driver.implicitly_wait(3)

    time.sleep(1.5)

    driver.execute_script('window.scrollTo(0, 800)') # 한번 스크롤
    time.sleep(3)

    last_height = driver.execute_script('return document.documentElement.scrollHeight') # 스크롤 전체 높이
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 10);")
    while True:
        driver.execute_script('window.scrollTo(0, document.documentElement.scrollHeight / 2);')
        time.sleep(3)
        driver.execute_script('window.scrollTo(0, document.documentElement.scrollHeight);') # 스크롤 다운
        time.sleep(3)

        new_height = driver.execute_script('return document.documentElement.scrollHeight') # 스크롤 다운 후 스크롤 높이 

        if new_height == last_height: # 댓글 마지막 페이지면 while문 벗어남
            break

        last_height = new_height
        time.sleep(1.5)
    print('스크롤 끝')
    time.sleep(5)   
    # 댓글 정보 불러오기
    comments_list = driver.find_elements(By.XPATH, '//*[@id="comment"]')

    comment_data = pd.DataFrame({'youtube_id':[],
                                'comment':[],
                                'like_num':[], 'comment_date':[]})
    html_s0 = driver.page_source
    html_s = BeautifulSoup(html_s0,'html.parser')

    comment0 = html_s.find_all('ytd-comment-renderer',{'class':'style-scope ytd-comment-thread-renderer'})
    print('댓글정보 불러오기 시작')
    for i in range(len(comment0)):
        

        # 댓글
        comment = comment0[i].find('yt-formatted-string',{'id':'content-text','class':'style-scope ytd-comment-renderer'}).text
        # 댓글의 좋아요 수
        try:
            aa = comment0[i].find('span',{'id':'vote-count-left'}).text
            #정규표현식으로 숫자만 추출하는 것은 정규표현식에 대한 공부를 더 한 뒤 해결
            #re.findall('[0-9]',aa)
            #"".join(re.findall('[0-9]',aa)) -> 리스트 내부의 문자열의 합
            like_num = "".join(re.findall('[0-9]',aa))
            like_num = int(like_num)
        except:
            like_num = 0

        # 댓글 작성자 id    
        bb = comment0[i].find('a',{'id':'author-text'}).find('span').text
        youtube_id = "".join(re.findall('[가-힣0-9a-zA-Z]',bb))

        # 댓글 작성 날짜
        comment_date = comment0[i].find('yt-formatted-string',{'class':'published-time-text style-scope ytd-comment-renderer'}).text
        
        # 데이터프레임에 데이터 저장
        comment_data.loc[len(comment_data), :] = [youtube_id, comment, like_num, comment_date]

    comment_data.index = range(len(comment_data))
    # 파일 저장


    comment_data.to_csv(f'./data/{youtuber}/{youtuber}_{video_num}_comment_data.csv')
    print(f'{youtuber}_{video_num}_comment_data.csv 저장완료')

In [ ]:
driver = webdriver.Chrome("chromedriver-win64/chromedriver.exe")

In [ ]:
for video_num in range(len(df)):
    url = df.iloc[video_num,1]
    driver.get(url)

    #페이지 Open 후 기다리는 시간
    time.sleep(5)
    
    # 영상 좋아요 수
    temp = driver.find_element(By.XPATH, '//*[@id="segmented-like-button"]/ytd-toggle-button-renderer/yt-button-shape/button')
    good = temp.get_attribute('aria-label')
    good_num = re.findall(r'\d{1,3}(?:,\d{3})*', good)
    if good_num:
        good_num = good_num[0].replace(',', '')  # 쉼표 제거
        df.iloc[video_num, 5] = good_num



    #down the scroll
    body = driver.find_element(By.TAG_NAME, 'body')
    last_page_height = driver.execute_script("return document.documentElement.scrollHeight")

    # max size 50의 Queue 생성
    # 0.1sec * 50 = 5sec 동안 Scroll 업데이트가 없으면 스크롤 내리기 종료
    szQ = Queue(4)
    enqueue_count = 0
    time.sleep(2)
    i = 0
    while True:
        # Scroll 내리기
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        if i == 0:
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 10);")
            time.sleep(3)
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 2);")
            time.sleep(3) 
        # Scroll Height를 가져오는 주기
        time.sleep(5)
        new_page_height = driver.execute_script("return document.documentElement.scrollHeight")
        
        # Queue가 꽉 차는 경우 스크롤 내리기 종료
        if(enqueue_count > szQ.maxsize):
            break
        
        # 첫 Loop 수행 (Queue가 비어있는 경우) 예외 처리
        if(szQ.isEmpty()) :
            szQ.enqueue(new_page_height)
            enqueue_count += 1
            
        # Queue에 가장 먼저 들어온 데이터와 새로 업데이트 된 Scroll Height를 비교함
        # 같으면 그대로 Enqueue, 다르면 Queue의 모든 Data를 Dequeue 후 새로운 Scroll Height를 Enqueue 함.    
        else :
            if(szQ.peek() == new_page_height) :
                szQ.enqueue(new_page_height)
                enqueue_count += 1
            else :
                szQ.enqueue(new_page_height)
                for z in range(enqueue_count) :
                    szQ.dequeue()
                enqueue_count = 1
        # time.sleep(3)
        if i > 10:
            driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(3)
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight / 3);")
            time.sleep(3)
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight * 2 / 3);")
            time.sleep(3)
            driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
            break
        else:
            i += 1